# Last Human on Earth - Colab render
T4 GPU render of the 60 s vertical short. Run cells top to bottom.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')
print(subprocess.run(['node', '-v'], capture_output=True, text=True).stdout)
print(subprocess.run(['npm', '-v'], capture_output=True, text=True).stdout)
print(sys.version)

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2 ffmpeg > /dev/null
print('deps ok')

In [ ]:
import os
if not os.path.isdir('/content/last-human-project'):
    !git clone https://github.com/shahrukhkathat199/last-human-project.git
%cd /content/last-human-project
!npm ci --legacy-peer-deps 2>&1 | tail -5
!npx remotion browser ensure

In [ ]:
%cd /content/last-human-project
import subprocess
cmd = ['npx','remotion','render','src/index.ts','LastHumanShort','out/last-human.mp4',
       '--codec=h264','--crf=20','--scale=1','--concurrency=2',
       '--gl=egl','--timeout=300000']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in p.stdout:
    tail.append(line.rstrip())
    if len(tail) > 40: tail.pop(0)
    if any(k in line for k in ('Rendered ','Encoded ','Error','error')):
        print(line.rstrip(), flush=True)
p.wait()
print('EXIT', p.returncode)
print('\n'.join(tail[-25:]))

In [ ]:
import os, subprocess
f = '/content/last-human-project/out/last-human.mp4'
print(os.path.getsize(f)/1e6, 'MB')
print(subprocess.run(['ffprobe','-v','error','-show_entries','stream=codec_type,codec_name,width,height,duration','-of','default=noprint_wrappers=1',f], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import files
files.download('/content/last-human-project/out/last-human.mp4')